# Socratic-OT | Phase 4: Multimodal Diagram Tutoring (VLM)

**Project:** Socratic-OT Multimodal AI Tutor  
**Team:** Vidhyadhari Bandaru, Richie Ilavarapu

> ⚡ **Enable GPU:** Runtime → Change runtime type → T4 GPU (required for local LLaVA)

### What this notebook does
1. **VLM Module** — identifies anatomical structures in uploaded diagrams using LLaVA-NeXT (local, open-source)
2. **Fallback** — GPT-4o Vision API (if LLaVA confidence is low or image is complex)
3. **Socratic Image Tutoring** — tutor asks about function/insertion/pathway BEFORE naming the structure
4. **Image-Text Linking** — links identified structure to relevant ChromaDB text chunks
5. **Blind Test Evaluation** — tests VLM on unseen anatomy diagrams, measures Structure ID Accuracy

### Architecture
```
Student uploads image
         │
         ▼
  [LLaVA-NeXT]  ──low confidence──→  [GPT-4o Vision fallback]
         │
    structure + topic
         │
         ▼
  [ChromaDB retrieval]  →  linked text context
         │
         ▼
  [Socratic question about function/insertion/pathway]
         │
         ▼ (after student answers)
  [Name the structure + explanation]
```

---
## Step 0 — Install Dependencies

> LLaVA-NeXT requires `transformers >= 4.36` and ~8GB VRAM. Colab T4 (16GB) is sufficient.

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -q chromadb sentence-transformers langchain langchain-groq groq
!pip install -q Pillow requests

In [ ]:
import os, json, torch
from google.colab import drive, userdata
drive.mount('/content/drive')

DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/Socratic_OT'
CHROMA_PERSIST     = os.path.join(DRIVE_PROJECT_ROOT, 'Data/chroma_db')
IMAGE_DIR          = os.path.join(DRIVE_PROJECT_ROOT, 'Data/images')
IMAGE_META_JSON    = os.path.join(DRIVE_PROJECT_ROOT, 'Data/image_metadata/image_metadata.json')

try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print('✅ Groq key from Colab Secrets')
except Exception:
    GROQ_API_KEY = 'gsk_YOUR_KEY_HERE'

# OpenAI key (optional — for GPT-4o fallback)
try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print('✅ OpenAI key from Colab Secrets (GPT-4o fallback enabled)')
except Exception:
    OPENAI_API_KEY = None
    print('ℹ️  No OpenAI key — will use LLaVA only (no GPT-4o fallback)')

os.environ['GROQ_API_KEY'] = GROQ_API_KEY
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB')

---
## Step 1 — Reload Retriever

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

embedder   = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
client     = chromadb.PersistentClient(path=CHROMA_PERSIST)
collection = client.get_collection('socratic_ot_textbook')
llm        = ChatGroq(model='llama-3.1-8b-instant', temperature=0.3, max_tokens=400)

with open(IMAGE_META_JSON) as f:
    image_metadata = json.load(f)

def retrieve(query, top_k=5):
    qemb = embedder.encode(query, normalize_embeddings=True).tolist()
    res  = collection.query(query_embeddings=[qemb], n_results=top_k,
                             include=['documents', 'metadatas', 'distances'])
    hits = []
    for doc, meta, dist, cid in zip(res['documents'][0], res['metadatas'][0],
                                     res['distances'][0], res['ids'][0]):
        hits.append({'chunk_id': cid, 'score': round(1-dist,4),
                     'section': meta.get('section',''), 'text': doc})
    merged = {}
    for h in hits:
        k = h['section']
        if k not in merged: merged[k] = h.copy()
        elif h['score'] > merged[k]['score']: merged[k]['score'] = h['score']
    return sorted(merged.values(), key=lambda x: x['score'], reverse=True)[:3]

print(f'✅ Retriever ready | {collection.count()} chunks')

---
## Step 2 — Load LLaVA-NeXT (Local VLM)

Using `llava-hf/llava-v1.6-mistral-7b-hf` — free, runs on Colab T4 with 4-bit quantization.

In [ ]:
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration, BitsAndBytesConfig
from PIL import Image
import requests, io

LLAVA_MODEL = 'llava-hf/llava-v1.6-mistral-7b-hf'

# 4-bit quantization to fit on Colab T4 (16GB)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print(f'Loading LLaVA-NeXT ({LLAVA_MODEL}) with 4-bit quantization...')
print('This takes ~3-4 minutes on first run...')

vlm_processor = LlavaNextProcessor.from_pretrained(LLAVA_MODEL)
vlm_model     = LlavaNextForConditionalGeneration.from_pretrained(
    LLAVA_MODEL,
    quantization_config=bnb_config,
    device_map='auto'
)

print('✅ LLaVA-NeXT loaded')

---
## Step 3 — VLM Inference Functions

In [ ]:
def run_llava(image: Image.Image, prompt: str, max_new_tokens: int = 256) -> str:
    """
    Run LLaVA-NeXT on an image with a text prompt.
    Returns the model's text response.
    """
    # LLaVA-NeXT uses a conversation template
    conversation = [
        {
            'role': 'user',
            'content': [
                {'type': 'image'},
                {'type': 'text', 'text': prompt}
            ]
        }
    ]
    formatted = vlm_processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs    = vlm_processor(images=image, text=formatted, return_tensors='pt').to('cuda')

    with torch.no_grad():
        output = vlm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None
        )

    # Decode only the new tokens (not the input)
    full   = vlm_processor.decode(output[0], skip_special_tokens=True)
    marker = '[/INST]'
    if marker in full:
        return full.split(marker)[-1].strip()
    return full.strip()


def identify_structure_llava(image: Image.Image) -> dict:
    """
    Ask LLaVA to identify the anatomical structure in the image.
    Returns: {structure, topic, confidence, description}
    """
    prompt = (
        'This is an anatomy or neuroscience diagram. '
        'Answer in this EXACT format:\n'
        'STRUCTURE: [name of the anatomical structure shown]\n'
        'TOPIC: [general topic: muscle / nerve / brain / skeleton / organ]\n'
        'CONFIDENCE: [High / Medium / Low]\n'
        'DESCRIPTION: [one sentence describing what is shown]\n'
        'Do not add any other text.'
    )
    raw = run_llava(image, prompt)

    import re
    result = {'raw': raw, 'structure': '', 'topic': '', 'confidence': '', 'description': ''}
    for field in ['STRUCTURE', 'TOPIC', 'CONFIDENCE', 'DESCRIPTION']:
        match = re.search(rf'{field}:\s*(.+?)(?=\n[A-Z]|$)', raw, re.DOTALL)
        result[field.lower()] = match.group(1).strip() if match else 'unknown'

    return result


print('✅ LLaVA inference functions defined')

---
## Step 4 — GPT-4o Vision Fallback (Optional)

In [ ]:
import base64

def image_to_base64(image: Image.Image) -> str:
    buffer = io.BytesIO()
    image.save(buffer, format='PNG')
    return base64.b64encode(buffer.getvalue()).decode('utf-8')


def identify_structure_gpt4o(image: Image.Image) -> dict:
    """
    GPT-4o Vision fallback — used when LLaVA confidence is Low.
    Requires OPENAI_API_KEY.
    """
    if not OPENAI_API_KEY:
        return {'structure': 'unknown', 'topic': 'unknown', 'confidence': 'N/A',
                'description': 'GPT-4o fallback not configured (no OpenAI key)'}

    import openai
    client_oai = openai.OpenAI(api_key=OPENAI_API_KEY)

    b64 = image_to_base64(image)
    response = client_oai.chat.completions.create(
        model='gpt-4o',
        messages=[
            {
                'role': 'user',
                'content': [
                    {'type': 'image_url',
                     'image_url': {'url': f'data:image/png;base64,{b64}', 'detail': 'high'}},
                    {'type': 'text',
                     'text': 'This is an anatomy diagram. Identify:\nSTRUCTURE: [name]\nTOPIC: [muscle/nerve/brain/skeleton/organ]\nCONFIDENCE: [High/Medium/Low]\nDESCRIPTION: [one sentence]'}
                ]
            }
        ],
        max_tokens=200
    )
    raw = response.choices[0].message.content

    import re
    result = {'raw': raw, 'structure': '', 'topic': '', 'confidence': '', 'description': ''}
    for field in ['STRUCTURE', 'TOPIC', 'CONFIDENCE', 'DESCRIPTION']:
        match = re.search(rf'{field}:\s*(.+?)(?=\n[A-Z]|$)', raw, re.DOTALL)
        result[field.lower()] = match.group(1).strip() if match else 'unknown'
    return result


print('✅ GPT-4o fallback defined (active:', OPENAI_API_KEY is not None, ')')

---
## Step 5 — VLM Module (Router with Fallback)

In [ ]:
class VLMModule:
    """
    Unified Vision-Language Module.
    Primary: LLaVA-NeXT (local, free)
    Fallback: GPT-4o (if confidence is Low and key is available)

    Workflow:
    1. Identify structure from image
    2. Retrieve linked text context from ChromaDB
    3. Generate a Socratic question about function/insertion/pathway
       WITHOUT naming the structure first
    """

    def analyze_image(self, image: Image.Image) -> dict:
        """Identify anatomical structure, using fallback if needed."""
        print('🔍 Running LLaVA-NeXT analysis...')
        result = identify_structure_llava(image)
        print(f'  LLaVA result: {result["structure"]} | Confidence: {result["confidence"]}')

        # Fallback to GPT-4o if low confidence
        if result['confidence'].lower() == 'low' and OPENAI_API_KEY:
            print('  → Confidence Low — falling back to GPT-4o Vision')
            result = identify_structure_gpt4o(image)
            print(f'  GPT-4o result: {result["structure"]} | Confidence: {result["confidence"]}')

        return result

    def retrieve_for_structure(self, structure: str, topic: str) -> str:
        """Retrieve textbook context relevant to the identified structure."""
        query  = f'{structure} {topic} anatomy function insertion'
        chunks = retrieve(query)
        return '\n\n'.join(c['text'] for c in chunks)

    def generate_socratic_question(self, structure: str, topic: str, context: str) -> str:
        """
        Generate a Socratic question about the structure WITHOUT naming it.
        Asks about function, insertion point, or pathway.
        """
        prompt = f"""You are a Socratic OT tutor. A student uploaded an anatomy diagram.
The diagram shows: {structure} (topic: {topic})

STRICT RULE: Do NOT name the structure in your question.
Instead, ask ONE question about its:
  - Primary function or action
  - Origin/insertion points (for muscles)
  - Pathway or innervation (for nerves)
  - Location relative to neighboring structures

Use this context to make your question grounded:
{context[:1200]}

Write only the question (1-2 sentences)."""

        resp = llm.invoke([HumanMessage(content=prompt)])
        return resp.content.strip()

    def name_and_explain(self, structure: str, context: str, student_answer: str) -> str:
        """
        After the student has answered, name the structure and provide
        a grounded explanation.
        """
        prompt = f"""You are a Socratic OT tutor. The student has answered your question.
Now reveal and explain the anatomical structure.

Structure: {structure}
Student's answer: {student_answer}
Textbook context: {context[:1200]}

Write a 3-4 sentence response that:
1. States the structure name clearly
2. Confirms or corrects the student's answer
3. Gives a grounded explanation from the textbook
4. Adds a brief OT clinical connection"""

        resp = llm.invoke([HumanMessage(content=prompt)])
        return resp.content.strip()

    def full_diagram_session(self, image: Image.Image) -> dict:
        """
        Run the complete image tutoring workflow. Returns a dict with
        identification results and the Socratic question to ask.
        """
        analysis = self.analyze_image(image)
        structure = analysis.get('structure', 'unknown structure')
        topic     = analysis.get('topic', 'anatomy')

        context   = self.retrieve_for_structure(structure, topic)
        question  = self.generate_socratic_question(structure, topic, context)

        return {
            'identified_structure': structure,
            'topic':               topic,
            'confidence':          analysis.get('confidence', ''),
            'description':         analysis.get('description', ''),
            'context':             context,
            'socratic_question':   question
        }


vlm = VLMModule()
print('✅ VLMModule defined')

---
## Step 6 — Test on Your 6 Anatomy Images

In [ ]:
import os
from IPython.display import display

# Get all images in the Data/images folder
image_files = [f for f in os.listdir(IMAGE_DIR)
               if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
print(f'Found {len(image_files)} images: {image_files}')


def test_single_image(filename: str):
    path  = os.path.join(IMAGE_DIR, filename)
    image = Image.open(path).convert('RGB')

    print(f'\n{'='*60}')
    print(f'Image: {filename}')
    display(image.resize((400, 300)))

    result = vlm.full_diagram_session(image)

    print(f'Identified structure : {result["identified_structure"]}')
    print(f'Topic                : {result["topic"]}')
    print(f'Confidence           : {result["confidence"]}')
    print(f'Description          : {result["description"]}')
    print(f'\nSocratic question (no name given):')
    print(f'  "{result["socratic_question"]}"')
    return result


# Test the first image
if image_files:
    test_single_image(image_files[0])

In [ ]:
# Test all 6 images and collect results
all_results = []

for fname in image_files:
    try:
        result = test_single_image(fname)
        result['filename'] = fname
        all_results.append(result)
    except Exception as e:
        print(f'Error on {fname}: {e}')

print(f'\n✅ Tested {len(all_results)}/{len(image_files)} images')

---
## Step 7 — Blind Test Evaluation

Compare VLM identifications against ground-truth labels from `image_metadata.json`.

In [ ]:
# Build ground-truth lookup
gt_by_filename = {}
for rec in image_metadata:
    fname = rec.get('file_name', '')
    if fname:
        gt_by_filename[fname.lower()] = {
            'structure': rec.get('structure_name', '').lower(),
            'aliases':   [a.lower() for a in rec.get('aliases', [])],
            'topic':     rec.get('topic', '').lower()
        }

print('Ground truth entries:', len(gt_by_filename))
print('Sample:', list(gt_by_filename.items())[:2])


def structure_matches(predicted: str, gt_structure: str, gt_aliases: list) -> bool:
    """
    Check if the predicted structure name matches ground truth or any alias.
    Uses substring matching to handle partial names.
    """
    pred_lower = predicted.lower()
    targets    = [gt_structure] + gt_aliases
    for target in targets:
        if target in pred_lower or pred_lower in target:
            return True
        # Check word overlap
        pred_words   = set(pred_lower.split())
        target_words = set(target.split())
        if len(pred_words & target_words) >= 1 and len(pred_words & target_words) / len(target_words) >= 0.5:
            return True
    return False


# Run blind test
print('\n=== BLIND TEST EVALUATION ===')
correct = 0
tested  = 0
eval_rows = []

for result in all_results:
    fname = result['filename'].lower()
    gt    = gt_by_filename.get(fname)

    if not gt:
        # Try partial filename match
        for gt_fname, gt_data in gt_by_filename.items():
            if gt_fname.split('.')[0] in fname or fname.split('.')[0] in gt_fname:
                gt = gt_data
                break

    if gt:
        tested += 1
        predicted = result['identified_structure']
        match     = structure_matches(predicted, gt['structure'], gt['aliases'])
        if match:
            correct += 1
        status = '✅' if match else '❌'
        print(f'{status} {result["filename"]}')
        print(f'   Predicted : {predicted}')
        print(f'   Expected  : {gt["structure"]} (aliases: {gt["aliases"][:3]})')
        eval_rows.append({
            'filename': result['filename'],
            'predicted': predicted,
            'expected': gt['structure'],
            'match': match,
            'confidence': result['confidence']
        })
    else:
        print(f'⚠️  No GT found for: {result["filename"]}')

accuracy = correct / tested if tested > 0 else 0
print(f'\n📊 Structure ID Accuracy: {correct}/{tested} = {accuracy:.2%}')
print(f'   Target: ≥ 80%')
print(f'   Status: {"✅ PASS" if accuracy >= 0.8 else "⚠️  Below target"}')

---
## Step 8 — Save Evaluation Results

In [ ]:
import json

EVAL_DIR = os.path.join(DRIVE_PROJECT_ROOT, 'Evaluation')
os.makedirs(EVAL_DIR, exist_ok=True)

vlm_eval = {
    'total_tested':  tested,
    'correct':       correct,
    'accuracy':      round(accuracy, 4),
    'target':        0.80,
    'pass':          accuracy >= 0.80,
    'details':       eval_rows
}

with open(os.path.join(EVAL_DIR, 'vlm_blind_test_results.json'), 'w') as f:
    json.dump(vlm_eval, f, indent=2)

print('✅ Saved: Evaluation/vlm_blind_test_results.json')
print()
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('PHASE 4 COMPLETE')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('  Primary VLM      : LLaVA-NeXT (4-bit, local)')
print('  Fallback VLM     : GPT-4o Vision (if Low confidence)')
print(f'  Structure ID     : {correct}/{tested} = {accuracy:.1%}')
print('  Socratic purity  : Structure named AFTER student answers')
print()
print('→ Next: Run 05_evaluation_gradio.ipynb')